In [1]:
import json
import random
import time
import uuid
from datetime import datetime

### Telemetry Helper

In [2]:
class Telemetry:

    def __init__(self, total_steps):
        self.trace_id = str(uuid.uuid4())

        self.total_steps = total_steps
        self.completed_steps = 0

        self.run_start_time = time.time()

        self.agent_start_times = {}

        self.completed_agents = 0

    def current_timestamp(self):
        return datetime.utcnow().isoformat() + "Z"

    def pipeline_percent_complete(self):
        return round(
            (self.completed_steps / self.total_steps) * 100,
            2
        )

    def throughput(self):
        elapsed = max(
            time.time() - self.run_start_time,
            0.001
        )

        return round(
            self.completed_steps / elapsed,
            2
        )

    def emit(self, payload):
        print(json.dumps(payload))

###  Agent

In [3]:
class Agent:

    def __init__(
            self,
            name,
            steps,
            fail_at_step=None):

        self.name = name
        self.steps = steps
        self.fail_at_step = fail_at_step

    def run(self, telemetry):

        span_id = str(uuid.uuid4())

        telemetry.agent_start_times[self.name] = time.time()

        # agent_started

        telemetry.emit({
            "timestamp": telemetry.current_timestamp(),
            "trace_id": telemetry.trace_id,
            "span_id": span_id,
            "event": "agent_started",
            "agent": self.name,
            "total_steps": self.steps
        })

        # Progress checkpoints
        checkpoints = set()

        checkpoints.add(max(1, round(self.steps * 0.25)))
        checkpoints.add(max(1, round(self.steps * 0.50)))
        checkpoints.add(max(1, round(self.steps * 0.75)))
        checkpoints.add(self.steps)

        # Agent Work

        for step in range(1, self.steps + 1):

            time.sleep(
                random.uniform(0.05, 0.20)
            )

            if (
                self.fail_at_step is not None
                and step == self.fail_at_step
            ):
                raise RuntimeError(
                    f"{self.name} failed at step {step}"
                )

            telemetry.completed_steps += 1

            if step in checkpoints:

                agent_progress = round(
                    (step / self.steps) * 100,
                    2
                )

                telemetry.emit({
                    "timestamp":
                        telemetry.current_timestamp(),

                    "trace_id":
                        telemetry.trace_id,

                    "span_id":
                        span_id,

                    "event":
                        "agent_progress",

                    "agent":
                        self.name,

                    "agent_percent_complete":
                        agent_progress,

                    "pipeline_percent_complete":
                        telemetry.pipeline_percent_complete(),

                    "throughput_steps_per_sec":
                        telemetry.throughput()
                })

        # agent_completed

        duration = round(
            time.time()
            - telemetry.agent_start_times[self.name],
            2
        )

        telemetry.completed_agents += 1

        telemetry.emit({
            "timestamp":
                telemetry.current_timestamp(),

            "trace_id":
                telemetry.trace_id,

            "span_id":
                span_id,

            "event":
                "agent_completed",

            "agent":
                self.name,

            "duration_seconds":
                duration,

            "pipeline_percent_complete":
                telemetry.pipeline_percent_complete(),

            "throughput_steps_per_sec":
                telemetry.throughput()
        })


### Orchestrator

In [4]:
class Orchestrator:

    def __init__(self, agents):

        self.agents = agents

        total_steps = sum(
            agent.steps
            for agent in agents
        )

        self.telemetry = Telemetry(total_steps)

    def run(self):

        failed_agent = None
        status = "success"

        try:

            for agent in self.agents:
                agent.run(self.telemetry)

        except Exception as ex:

            status = "failed"
            failed_agent = agent.name

            self.telemetry.emit({

                "timestamp":
                    self.telemetry.current_timestamp(),

                "trace_id":
                    self.telemetry.trace_id,

                "span_id":
                    str(uuid.uuid4()),

                "event":
                    "agent_failed",

                "agent":
                    agent.name,

                "error":
                    str(ex),

                "pipeline_percent_complete":
                    self.telemetry.pipeline_percent_complete(),

                "throughput_steps_per_sec":
                    self.telemetry.throughput()
            })

        finally:

            total_duration = round(
                time.time()
                - self.telemetry.run_start_time,
                2
            )

            self.telemetry.emit({

                "timestamp":
                    self.telemetry.current_timestamp(),

                "trace_id":
                    self.telemetry.trace_id,

                "span_id":
                    None,

                "event":
                    "run_summary",

                "status":
                    status,

                "duration_seconds":
                    total_duration,

                "agents_completed":
                    self.telemetry.completed_agents,

                "failed_agent":
                    failed_agent,

                "pipeline_percent_complete":
                    self.telemetry.pipeline_percent_complete(),

                "throughput_steps_per_sec":
                    self.telemetry.throughput()
            })



### Main

In [5]:
def main():

    agents = [

        Agent("Planner", 3),

        Agent("Researcher", 6),

        Agent("Writer", 4),

        Agent("Reviewer", 2)

    ]

    Orchestrator(agents).run()


if __name__ == "__main__":
    main()

/tmp/ipykernel_2185/2131353610.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat() + "Z"


{"timestamp": "2026-06-24T06:09:53.263417Z", "trace_id": "a59c7a44-0feb-409b-8714-0da218f6e639", "span_id": "6a217ad7-a662-4a16-8de9-cc6a99cba10b", "event": "agent_started", "agent": "Planner", "total_steps": 3}
{"timestamp": "2026-06-24T06:09:53.458527Z", "trace_id": "a59c7a44-0feb-409b-8714-0da218f6e639", "span_id": "6a217ad7-a662-4a16-8de9-cc6a99cba10b", "event": "agent_progress", "agent": "Planner", "agent_percent_complete": 33.33, "pipeline_percent_complete": 6.67, "throughput_steps_per_sec": 5.12}
{"timestamp": "2026-06-24T06:09:53.596496Z", "trace_id": "a59c7a44-0feb-409b-8714-0da218f6e639", "span_id": "6a217ad7-a662-4a16-8de9-cc6a99cba10b", "event": "agent_progress", "agent": "Planner", "agent_percent_complete": 66.67, "pipeline_percent_complete": 13.33, "throughput_steps_per_sec": 6.0}
{"timestamp": "2026-06-24T06:09:53.656683Z", "trace_id": "a59c7a44-0feb-409b-8714-0da218f6e639", "span_id": "6a217ad7-a662-4a16-8de9-cc6a99cba10b", "event": "agent_progress", "agent": "Planner",